# Nota Secreta — A2A / LLM (Colab)

Notebook pronto para rodar o jogo **Nota Secreta** no Google Colab.

O projeto sobe, como subprocessos HTTP:

- um **serviço LLM** centralizado (`llm_service.py`);
- o **Game Master** (`game_master.py`);
- **1 agente estratégico** (`llm_agent.py`) + **5 agentes aleatórios** (`random_agent.py`).

> **Por que rodamos via `!python run_game.py ...` e não via `import`?**
> O `run_game.py` usa `asyncio.run(...)`, que conflita com o event loop já ativo do Jupyter/Colab. Rodar como processo de shell evita esse conflito e mantém o código original intacto.

**Fluxo recomendado:** seções 1 → 2 → 3 → 4 (mock). A seção 5 (modelo real) é opcional e mais pesada.

## 1. Clonar o repositório

Troque `SEU_USUARIO` pelo seu usuário do GitHub (e o nome do repo, se for diferente).

In [ ]:
REPO_URL = "https://github.com/SEU_USUARIO/nota-secreta-a2a.git"

import os
repo_dir = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo_dir):
    !git clone $REPO_URL
%cd $repo_dir
!ls -la

## 2. Instalar dependências (modo mock)

Para validar a arquitetura em modo mock **não** é preciso instalar `llama-cpp-python` (que é pesado e compila). Instalamos só o essencial:

In [ ]:
!pip install -q "fastapi>=0.110" "uvicorn>=0.29" "pydantic>=2.6" "aiohttp>=3.9" "pytest>=8.0"

## 3. Rodar os testes

Testes de pontuação (`tests/test_scoring.py`):

In [ ]:
!python -m pytest tests -v

## 4. Partida em modo mock

Roda uma partida completa **sem modelo real** (rápido, ótimo para validar a infraestrutura e o protocolo do agente). O serviço LLM responde com um texto fixo.

Ao final, o caminho do log é mostrado no terminal e salvo em `logs/`.

In [ ]:
!python run_game.py --force-mock

Variante: **6 agentes estratégicos** em modo mock (todos usando `llm_agent.py`):

In [ ]:
!python run_game.py --all-strategic --force-mock

## 5. (Opcional) Partida com modelo real (GGUF)

Esta seção usa um modelo LLM real (`Phi-3.5-mini-instruct`, ~2,4 GB) via `llama-cpp-python`.

**Atenção:**
- A instalação do `llama-cpp-python` **compila** e pode levar alguns minutos.
- O download do modelo (~2,4 GB) também demora.
- Para acelerar a inferência, use um runtime com **GPU** no Colab (`Ambiente de execução → Alterar tipo de ambiente de execução → GPU`).

### 5.1. Instalar o `llama-cpp-python`

In [ ]:
# CPU (sempre funciona). Para GPU, veja a célula alternativa abaixo.
!pip install -q "llama-cpp-python>=0.2.70"

In [ ]:
# (Alternativa GPU) Instala wheel com suporte a CUDA — use só se o runtime tiver GPU.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q --force-reinstall --no-cache-dir "llama-cpp-python>=0.2.70"

### 5.2. Baixar o modelo GGUF

In [ ]:
import os
os.makedirs('models', exist_ok=True)
MODEL_PATH = 'models/phi-3.5-mini-instruct-q4_k_m.gguf'
MODEL_URL = 'https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf'
if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O $MODEL_PATH $MODEL_URL
!ls -lh models/

### 5.3. Rodar a partida com o modelo real

`--llm-max-concurrency 1` evita sobrecarregar o modelo local (uma requisição por vez).

In [ ]:
!python run_game.py --model models/phi-3.5-mini-instruct-q4_k_m.gguf --llm-max-concurrency 1

## 6. Ler o log da última partida

Transforma o log JSON mais recente numa visualização legível (narrador, dicas, cartas jogadas, votos e evolução da pontuação).

In [ ]:
import glob, os
logs = sorted(glob.glob('logs/*.json'), key=os.path.getmtime)
if logs:
    ultimo = logs[-1]
    print('Log mais recente:', ultimo)
    !python render_log_readable.py "$ultimo"
else:
    print('Nenhum log encontrado — rode uma partida nas seções 4 ou 5 primeiro.')